In [2]:
!pip install shapely
import json
import os
from shapely.geometry import Polygon

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 125.7 MB/s eta 0:00:00

[notice] A new release of pip is available: 25.1.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


In [5]:
def labelstudio_to_coco(ls_json_path, output_coco_path):
    # Load Label Studio JSON
    with open(ls_json_path, 'r') as f:
        ls_data = json.load(f)
        
    coco_data = {
        "images": [],
        "annotations": [],
        "categories": []
    }
    
    category_map = {}
    category_id_counter = 1
    annotation_id_counter = 1
    
    for task_idx, task in enumerate(ls_data):
        # 1. Process Image Information
        # Extract filename from URL or storage path
        file_upload = task.get("file_upload", "")
        # Fallback to data key if file_upload is empty
        img_url = task.get("data", {}).get("image", file_upload)
        filename = os.path.basename(img_url)
        
        # Label Studio saves original dimensions inside the results, 
        # but we need a default placeholder if a task has no annotations.
        width = None
        height = None
        
        task_annotations = task.get("annotations", [])
        if not task_annotations:
            continue
            
        # Standard Label Studio format stores results inside the first annotation element
        results = task_annotations[0].get("result", [])
        
        # Pull dimensions from the first valid result item if possible
        for result in results:
            if "original_width" in result and "original_height" in result:
                width = result["original_width"]
                height = result["original_height"]
                break
        
        # Skip images if dimensions are completely missing
        if width is None or height is None:
            print(f"Skipping task {task.get('id')} due to missing original dimensions.")
            continue

        image_id = task.get("id", task_idx)
        coco_data["images"].append({
            "id": image_id,
            "file_name": filename,
            "width": width,
            "height": height
        })
        
        # 2. Process Segmentation Results
        for result in results:
            if result.get("type") != "polygonlabels":
                continue # Skip bounding boxes or keypoints; we derive them from polygons
                
            value = result.get("value", {})
            points = value.get("points", [])
            label = value.get("polygonlabels", [None])[0]
            
            if not label or len(points) < 3:
                continue
                
            # Handle Category Mapping
            if label not in category_map:
                category_map[label] = category_id_counter
                coco_data["categories"].append({
                    "id": category_id_counter,
                    "name": label,
                    "supercategory": "object"
                })
                category_id_counter += 1
                
            cat_id = category_map[label]
            
            # Label Studio stores points as percentages (0-100). 
            # We must convert them back to absolute pixel coordinates.
            absolute_polygon = []
            for pt in points:
                abs_x = (pt[0] / 100.0) * width
                abs_y = (pt[1] / 100.0) * height
                absolute_polygon.extend([abs_x, abs_y])
                
            # Reshape into pairs for Shapely logic
            poly_pairs = [(absolute_polygon[i], absolute_polygon[i+1]) for i in range(0, len(absolute_polygon), 2)]
            
            try:
                poly_geom = Polygon(poly_pairs)
                if not poly_geom.is_valid:
                    poly_geom = poly_geom.buffer(0) # Clean up self-intersecting geometries
                    
                area = poly_geom.area
                # Get bounding box: [minx, miny, maxx, maxy]
                minx, miny, maxx, maxy = poly_geom.bounds
                bbox = [minx, miny, maxx - minx, maxy - miny] # Convert to [x, y, w, h]
            except Exception as e:
                print(f"Geometry error on task {image_id}: {e}")
                continue

            # Build COCO annotation entry
            coco_data["annotations"].append({
                "id": annotation_id_counter,
                "image_id": image_id,
                "category_id": cat_id,
                "segmentation": [absolute_polygon],
                "bbox": bbox,
                "area": area,
                "iscrowd": 0
            })
            annotation_id_counter += 1

    # Save to file
    with open(output_coco_path, 'w') as f:
        json.dump(coco_data, f, indent=4)
    print(f"Successfully converted data! Saved to: {output_coco_path}")

# Example usage:
labelstudio_to_coco('../annotations/export_19_project.json', '../annotations/coco_new.json')


Skipping task 6908 due to missing original dimensions.
Skipping task 6910 due to missing original dimensions.
Skipping task 6912 due to missing original dimensions.
Skipping task 6918 due to missing original dimensions.
Skipping task 6919 due to missing original dimensions.
Skipping task 6920 due to missing original dimensions.
Skipping task 6921 due to missing original dimensions.
Skipping task 6924 due to missing original dimensions.
Skipping task 6925 due to missing original dimensions.
Skipping task 6927 due to missing original dimensions.
Skipping task 6928 due to missing original dimensions.
Skipping task 6929 due to missing original dimensions.
Skipping task 6932 due to missing original dimensions.
Skipping task 6940 due to missing original dimensions.
Skipping task 6946 due to missing original dimensions.
Skipping task 6950 due to missing original dimensions.
Skipping task 6954 due to missing original dimensions.
Skipping task 6955 due to missing original dimensions.
Skipping t